In [1]:
###### IMPORT STATEMENTS #####
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
import torchaudio
import json
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import time
import re
from google.colab import drive
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Union
import sys


In [3]:
!ls /content/drive/MyDrive/thesis

__pycache__  results  SSM_model.py  SSM_TU  TIMITDataset.py  TIMIT.zip


In [2]:
###### loading model WITH Google COLAB #####
drive.mount('/content/drive', force_remount=True)
drive_root_directory = os.path.join(os.getcwd(), "drive/MyDrive")
thesis_directory = os.path.join(drive_root_directory, "thesis")
results_directory = os.path.join(thesis_directory, "results")

sys.path.append("/content/drive/MyDrive/thesis") # append the path to the thesis for import of SSM_model.py 

# import the model and dataset files from drive
from SSM_model import SSMPhonemeModel
from TIMITDataset import TIMITDataset
from utils import ctc_greedy_decode

print("drive_root_directory:", drive_root_directory)
print("thesis_directory:", thesis_directory)
print("results_directory:", results_directory)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA detected -> GPU is used")
else:
    device = torch.device("cpu")
    print("No GPU detected -> CPU is used")

Mounted at /content/drive
drive_root_directory: /content/drive/MyDrive
thesis_directory: /content/drive/MyDrive/thesis
results_directory: /content/drive/MyDrive/thesis/results
No GPU detected -> CPU is used


In [3]:
######### UNZIP the TIMIT dataset #########

# unzip the data
if not os.path.exists("/content/TIMIT"):  # or your dataset folder
    !unzip -q /content/drive/MyDrive/thesis/TIMIT.zip -d /content/TIMIT
else:
    print("TIMIT folder already exists in /content/TIMIT folder")

# check the content of the main directory
!ls /content/TIMIT

####### PATHS and VARIABLE CONSTANTS #######
colab = True # decides whether colab or local pc used

project_root_dir = os.getcwd()
if colab:
    data_root_dir = os.path.join(project_root_dir, "TIMIT")
else:
    data_root_dir = os.path.join(project_root_dir, "data")

vocabulary_dir = os.path.join(project_root_dir, "vocabulary")
phoneme_to_idx_vocab_filename = "phoneme_to_idx.json"
idx_to_phoneme_vocab_filename = "idx_to_phoneme.json"
train_data_csv = os.path.join(data_root_dir, "train_data.csv")
test_data_csv = os.path.join(data_root_dir, "test_data.csv")

print(f"Project root directory: {project_root_dir}")
print(f"Data root directory: {data_root_dir}")
print(f"Vocabulary directory: {vocabulary_dir}")
print(f"train_data.csv: {train_data_csv}")
print(f"test_data.csv: {test_data_csv}")

data	      README.DOC    test_data.csv  TIMITDIC.TXT
PHONCODE.DOC  SPKRINFO.TXT  TESTSET.DOC    train_data.csv
PROMPTS.TXT   SPKRSENT.TXT  TIMITDIC.DOC
Project root directory: /content
Data root directory: /content/TIMIT
Vocabulary directory: /content/vocabulary
train_data.csv: /content/TIMIT/train_data.csv
test_data.csv: /content/TIMIT/test_data.csv


In [18]:
######## CONFIGURATION ########
EXPERIMENT_NAME = "result_1"
MODEL_FILENAME = "ssm_model_state_dict.pt"
CONFIG_FILENAME = "run_config.json"

######## BUILD PATHS ########
result_dir = os.path.join(results_directory, EXPERIMENT_NAME)
print(results_directory)
model_path = os.path.join(result_dir, MODEL_FILENAME)
config_path = os.path.join(result_dir, CONFIG_FILENAME)

print(f"Loading model from: {model_path}")
print(f"Loading config from: {config_path}")

######## LOAD CONFIG ########
######## LOAD CONFIG ########
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

######## USE SAVED VOCAB ########
phoneme_to_idx_vocab = config["phoneme_to_idx_vocab"]
idx_to_phoneme_vocab = {int(v): k for k, v in phoneme_to_idx_vocab.items()}
blank_idx = config["blank_index"]

######## INITIALIZE MODEL ########
model = SSMPhonemeModel(
    d_in=config["ssm_model_d_in"],
    d_state=config["ssm_d_state"],
    vocab_size=config["ssm_vocab_size"],
    dropout=config["ssm_dropout"]
)

######## LOAD SAVED WEIGHTS ########
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)

######## PREPARE MODEL ########
model.to(device)
model.eval()

print("✅ Model loaded successfully!")

/content/drive/MyDrive/thesis/results
Loading model from: /content/drive/MyDrive/thesis/results/result_1/ssm_model_state_dict.pt
Loading config from: /content/drive/MyDrive/thesis/results/result_1/run_config.json
✅ Model loaded successfully!


In [19]:
##### DEFINE DATASET AND TRANSFORM ####
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=config["mel_transform_sample_rate"],
    n_fft=config["mel_transform_nfft"], # FFT window size => how many samples are used for frequency analysis
    hop_length=config["mel_transform_hop_length"], # step size between windows => at 16kHz a hop length of 160 means a stepsize of 10ms
    win_length=config["mel_transform_win_length"], # Actual window size applied before FFT => at 16kHz a window size of 400 means 25ms window length
    n_mels=config["mel_transform_n_mels"] # Number of frequency bins after Mel conversion => number of frequency elements in each window AFTER conversion
)

# create the test dataset object
test_dataset = TIMITDataset(
    csv_file=test_data_csv,
    train_or_test="TEST",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=mel_transform
)

Filtering dataframe to only include TEST samples


In [26]:
###### PREPARE ONE TEST SAMPLE FOR INFERENCE ######

# select sample index
sample_index = 0   # change this to try different samples

# load one sample from test dataset
features, labels = test_dataset[sample_index]

print("Feature shape [T, 80]:", features.shape)
print("Label shape [L]:", labels.shape)

Feature shape [T, 80]: torch.Size([397, 80])
Label shape [L]: torch.Size([41])


In [27]:
###### SAMPLE METADATA ######
sample_info = test_dataset.samples[sample_index]

print("Base name:", sample_info["base_name"])
print("Audio path:", sample_info["audio_path"])
print("Phoneme path:", sample_info["phoneme_path"])

Base name: TEST/DR1/FAKS0/SA1
Audio path: /content/TIMIT/data/TEST/DR1/FAKS0/SA1.WAV
Phoneme path: /content/TIMIT/data/TEST/DR1/FAKS0/SA1.PHN


In [28]:
###### BUILD MODEL INPUT ######
# add batch dimension: [T, 80] -> [1, T, 80]
X_inference = features.unsqueeze(0).to(device)

# length tensor -> useful later for CTC / decoding
input_length = torch.tensor([features.shape[0]], dtype=torch.long)

print("Model input shape [1, T, 80]:", X_inference.shape)
print("Input length:", input_length)

Model input shape [1, T, 80]: torch.Size([1, 397, 80])
Input length: tensor([397])


In [29]:
###### FORWARD PASS ######
with torch.inference_mode():
    logits = model(X_inference)   # [1, T, vocab_size]

print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 397, 62])


In [30]:
###### FRAMEWISE PREDICTIONS ######
pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

print("First 50 predicted ids:", pred_ids[:50])

from collections import Counter

counter = Counter(pred_ids)
print("Most common predicted ids:", counter.most_common(10))
print("Unique predicted ids:", sorted(set(pred_ids)))

First 50 predicted ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 28, 28, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Most common predicted ids: [(0, 351), (33, 4), (49, 3), (58, 3), (4, 3), (59, 3), (28, 2), (50, 2), (30, 2), (14, 2)]
Unique predicted ids: [0, 1, 4, 8, 13, 14, 16, 22, 23, 26, 27, 28, 30, 31, 33, 35, 36, 39, 48, 49, 50, 51, 52, 56, 58, 59]


In [31]:
###### CONVERT TO PHONEMES ######
# decode the predicted id-s because of CTC
decoded_ids = ctc_greedy_decode(pred_ids, blank_idx=0)

# decode the phonemes based on the idx_to_phoneme_vocab vocabulary file
before_decoding_raw_phonemes = [idx_to_phoneme_vocab[i] for i in pred_ids]
decoded_phonemes = [idx_to_phoneme_vocab[i] for i in decoded_ids]

# get the target phonemes for comparison
target_ids = labels.cpu().tolist()
target_phonemes = [idx_to_phoneme_vocab[i] for i in target_ids]

print("Predicted phonemes before CTC decoding:")
print(before_decoding_raw_phonemes)

print("\nPredicted phonemes:")
print(decoded_phonemes)

print("\nTarget phonemes:")
print(target_phonemes)

Predicted phonemes before CTC decoding:
['<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', 'h#', 'h#', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', 'sh', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', 'iy', 'iy', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', '<blank>', 'hv', 'hv', '<blank>', '<blank>', '<blank>', '<blank>', '<bl